In [1]:
# Load and save global mortality in single files - CHECK ATTRIBUTES

In [2]:
import os
import glob
import xarray as xr
from utils.utils import get_scenario_config

In [5]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "G6-1.5K"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]
dates = f"{years.start}_{years.stop - 1}"

MORT_DIR = f"/glade/work/awells/air_quality/{model}/mortality/ozone/global/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/mortality/ozone/"

for ens_num in ensemble_members:
    print(f"Processing ensemble number {ens_num:02d}")

    files = f"Global_mortality_stats_{model}_{scenario}_{ens_num:02d}_*.nc"
    file_path = os.path.join(MORT_DIR, files)

    ds = xr.open_mfdataset(
        sorted(glob.glob(file_path)),
        combine="nested",
        concat_dim="year")
    ds = ds.assign_coords(year=years[:-1])  # Last year removed from OSDMA8

    description = ("Global mortality (COPD) due to ozone "
                   "statistics: including mean, median, "
                   "and the 95% CI - scripts by A.F. Wells (2025)")
    ds.attrs["description"] = description
    ds.attrs["model"] = model
    ds.attrs["scenario"] = scenario
    ds.attrs["ensemble_number"] = ens_num

    out_file = f"Global_mortality_stats_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)
    ds.to_netcdf(out_path)

print("All processing complete.")

Processing ensemble number 01
Processing ensemble number 02
Processing ensemble number 03
All processing complete.
